In [ ]:
# Loome SparkSession-i 
# spark.jars.packages laeb vajalikud JAR-id Maven Central-ist Sparki käivitumisel.
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("metalfab")
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.11")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}")

In [ ]:
import os

# Andmebaasi ühendus loetakse keskkonnamuutujatest (sama leping nagu dags/elering_ingest.py).
# Vaikeväärtused vastavad arenduskeskkonnale; compose.yml edastab need konteineri env'i.
DB_HOST = os.environ.get("POSTGRES_HOST")
DB_PORT = os.environ.get("POSTGRES_PORT")
DB_NAME = os.environ.get("POSTGRES_DB")
DB_USER = os.environ.get("POSTGRES_USER")
DB_PASSWORD = os.environ.get("POSTGRES_PASSWORD")

def write_to_postgres(batch_df, batch_id):
    db_url = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
    db_properties = {
       "user": DB_USER,
       "password": DB_PASSWORD,
       "driver": "org.postgresql.Driver"
        }
    # Kirjutame selle konkreetse partii andmebaasi
    batch_df.write.jdbc(
       url=db_url,
       table="bronze.raw_factory_data",
       mode="append",
       properties=db_properties
       )
    print(f"Batch {batch_id} kirjutatud andmebaasi.")

processed_df = spark.readStream \
    .schema("dept STRING, machine STRING, tag STRING, timestamp TIMESTAMP, topic STRING, value DOUBLE") \
    .json("../data/lake/*/month=06/*/*/*/*")

# 4. Käivita striim
query = processed_df.writeStream \
.foreachBatch(write_to_postgres) \
.option("checkpointLocation", "./checkpoints/postgres_stream") \
.start()

In [ ]:
query.status

In [ ]:
#query.stop()